### NFL Draft Prediction

This notebook contains an advanced machine learning pipeline leveraging position-specific median imputation, custom feature interaction mapping, and a blended ensemble strategy designed for high-ranking leaderboard placement.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

## 1. Feature Engineering Formulas

In [ ]:
def engineer_features(df):
    df = df.copy()
    # Size-Adjusted Speed Metric (Speed Score)
    if 'Weight' in df.columns and 'Sprint_40yd' in df.columns:
        df['Speed_Score'] = (df['Weight'] * 2.20462) / (df['Sprint_40yd'] ** 4)
    # Body Mass Index (BMI)
    if 'Weight' in df.columns and 'Height' in df.columns:
        df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    # Lower Body Explode Index
    if 'Vertical_Jump' in df.columns and 'Broad_Jump' in df.columns:
        df['Explosive_Index'] = df['Vertical_Jump'] + df['Broad_Jump']
    # Change of Direction Agility Index
    if 'Agility_3cone' in df.columns and 'Shuttle' in df.columns:
        df['Agility_Index'] = df['Agility_3cone'] + df['Shuttle']
    return df

## 2. Advanced Preprocessing & Imputation Pipeline

In [ ]:
train = pd.read_csv('Ctrain.csv')
test = pd.read_csv('Ctest.csv')
combined = pd.concat([train.drop('Drafted', axis=1), test], axis=0).reset_index(drop=True)

# Advanced Position-wise Imputation
num_cols = [c for c in combined.select_dtypes(include=[np.number]).columns if c not in ['Id', 'Year']]
for col in num_cols:
    combined[col] = combined.groupby('Position')[col].transform(lambda x: x.fillna(x.median()))
    combined[col] = combined[col].fillna(combined[col].median())

combined = engineer_features(combined)

cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
for col in cat_cols:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

X_train = combined.iloc[:len(train)].copy()
X_test = combined.iloc[len(train):].copy()
y_train = train['Drafted']
features = [c for c in X_train.columns if c not in ['Id', 'Year']]

## 3. 10-Fold Validation & Out-Of-Fold Ensembling

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1337)
oof_hgb, test_hgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_et, test_et = np.zeros(len(X_train)), np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx][features], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx][features], y_train.iloc[val_idx]
    
    m1 = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.02, max_depth=6, l2_regularization=1.5, random_state=1337+fold)
    m1.fit(X_tr, y_tr)
    oof_hgb[val_idx] = m1.predict_proba(X_val)[:, 1]
    test_hgb += m1.predict_proba(X_test[features])[:, 1] / skf.n_splits
    
    m2 = ExtraTreesClassifier(n_estimators=250, max_depth=10, min_samples_split=5, random_state=42+fold, n_jobs=-1)
    m2.fit(X_tr, y_tr)
    oof_et[val_idx] = m2.predict_proba(X_val)[:, 1]
    test_et += m2.predict_proba(X_test[features])[:, 1] / skf.n_splits

# Blended Optimization
best_score, best_w = 0, 0
for w in np.linspace(0, 1, 101):
    score = roc_auc_score(y_train, (w * oof_hgb) + ((1 - w) * oof_et))
    if score > best_score:
        best_score, best_w = score, w

print(f"Optimized Cross-Validation ROC-AUC: {best_score:.5f} using Weight: {best_w}")
final_preds = (best_w * test_hgb) + ((1 - best_w) * test_et)
submission = pd.DataFrame({'Id': test['Id'], 'Drafted': final_preds})
submission.to_csv('submission.csv', index=False)